 storm risk model (Colab)

This notebook trains the **storm risk model** for project, the number shown as "Storm risk 78%" in engineer mode.

**The question it answers:** *Will a tropical storm or hurricane center pass within 150 km of here in the next 12 hours?* It only looks at **air pressure history**, because that is the only weather sensor the box has.

**What it does, step by step:**
1. Downloads real pressure records from NOAA weather buoys and coastal stations around Florida (NDBC)
2. Downloads every Atlantic storm track (HURDAT2)
3. Labels every hour: 1 = a storm is coming within 12 h, 0 = not
4. Builds 6 pressure features (the same ones the box can compute)
5. Trains a **logistic regression** and a **decision tree**, then compares them fairly
6. Exports a tiny JSON model + a pure-Python runtime (the Pi does **not** need scikit-learn)
7. Exports real storms as replay CSVs for the demo

In [ ]:
# STEP 0: imports (all of these are already installed in Colab)
import os, io, re, gzip, json, math, time, shutil, urllib.request, urllib.error
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib, sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

for d in ["data/ndbc", "out/models", "out/replay", "out/figures"]:
    os.makedirs(d, exist_ok=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

## Step 1: settings

These are the only numbers you should need to change.

- **Stations:** NOAA buoys and coastal stations around Florida. Any station without data is skipped automatically, so it's safe to add more.
- **Years:** train on older storms, pick the alarm threshold on middle years, and test on storms the model has **never seen** (2022+: Ian, Nicole, Idalia, Debby, Helene, Milton…). Splitting by year means one storm can never be in both train and test.
- **Season only:** we keep June–November (hurricane season), which is when the box matters and halves the data.

In [ ]:
# STEP 1: settings
STATIONS = [
    # Gulf side buoys
    "42036", "42039", "42003", "42013", "42022", "42023",
    # Atlantic side buoys
    "41009", "41010",
    # coastal stations (C-MAN / NOS)
    "venf1", "cdrf1", "sauf1", "lkwf1", "kywf1", "smkf1", "fwyf1", "mlrf1",
    "apcf1", "pacf1", "npsf1", "vcaf1", "cwbf1",
]
YEARS       = range(2005, 2026)
TRAIN_YEARS = range(2005, 2019)
VAL_YEARS   = range(2019, 2022)   # used only to choose the alarm threshold
TEST_YEARS  = range(2022, 2026)   # storms the model never saw

RADIUS_KM   = 150     # "storm close" = center within this distance
HORIZON_H   = 12      # ...within the next 12 hours
NORMAL_KM   = 500     # "normal weather" = no storm within 500 km for +/- 48 h
SEASON_MONTHS = range(6, 12)       # June-November
MAX_DEPTH   = 3       # small tree = easy to draw and explain
MIN_LEAF    = 100     # each tree leaf needs at least this many hours
FA_PER_WEEK_TARGET = 1.0           # at most 1 false alarm per week of normal weather
P_MIN, P_MAX = 850.0, 1090.0       # anything outside this is a bad reading

HURDAT_FALLBACK = "https://www.nhc.noaa.gov/data/hurdat/hurdat2-1851-2024-040425.txt"

## Step 2: download helpers + station locations

We need each station's latitude/longitude to measure how far each storm was. NOAA publishes one table with every station's location.

In [ ]:
# STEP 2: download helper + station coordinates
def fetch(url, retries=2, timeout=60):
    # Returns the file bytes, or None if the file does not exist (404).
    err = None
    for attempt in range(retries + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "StormBox-student-project"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            err = e
        except Exception as e:
            err = e
        time.sleep(1 + attempt)
    print("  ! failed:", url, err)
    return None

def load_station_coords():
    raw = fetch("https://www.ndbc.noaa.gov/data/stations/station_table.txt")
    if raw is None:
        raise RuntimeError("Could not download the NDBC station table")
    coords = {}
    for line in raw.decode("latin-1").splitlines():
        if line.startswith("#"):
            continue
        parts = line.split("|")
        if len(parts) < 7:
            continue
        m = re.match(r"\s*([\d.]+)\s+([NS])\s+([\d.]+)\s+([EW])", parts[6])
        if m:
            lat = float(m[1]) * (1 if m[2] == "N" else -1)
            lon = float(m[3]) * (1 if m[4] == "E" else -1)
            coords[parts[0].strip().lower()] = (lat, lon)
    return coords

COORDS = load_station_coords()
STATION_POS = {s.lower(): COORDS[s.lower()] for s in STATIONS if s.lower() in COORDS}
print(f"{len(STATION_POS)} of {len(STATIONS)} stations found in the NOAA table")
for s, (la, lo) in STATION_POS.items():
    print(f"  {s:6s} {la:7.3f}, {lo:8.3f}")

## Step 3: download pressure history (NDBC)

Each station has one file per year. Files are cached in `data/ndbc/`, so re-running is fast. Missing pressure is written as `9999` by NOAA; we drop anything outside 850–1090 hPa.

Then we turn the 10-minute readings into **one value per hour** (the average of the hour that just ended, so the model never peeks into the future) and fill gaps of up to 2 hours.

In [ ]:
# STEP 3: NDBC pressure files
def parse_ndbc(text):
    # Parses one NDBC 'stdmet' yearly file. Handles the old (BAR) and new (PRES) formats.
    lines = [l for l in text.splitlines() if l.strip()]
    if not lines:
        return pd.Series(dtype=float)
    header = lines[0].lstrip("#").split()
    rows = [l.split() for l in lines[1:] if not l.startswith("#")]
    rows = [r for r in rows if len(r) == len(header)]
    pcol = "PRES" if "PRES" in header else ("BAR" if "BAR" in header else None)
    if not rows or pcol is None:
        return pd.Series(dtype=float)
    df = pd.DataFrame(rows, columns=range(len(header)))
    num = lambda i: pd.to_numeric(df[i], errors="coerce")
    year = num(0)
    year = year.where(year >= 100, year + 1900)
    minute = num(4) if header[4] == "mm" else pd.Series(0, index=df.index)
    ts = pd.to_datetime(dict(year=year, month=num(1), day=num(2), hour=num(3), minute=minute),
                        errors="coerce", utc=True)
    p = num(header.index(pcol))
    s = pd.Series(p.values, index=ts).dropna()
    s = s[(s > P_MIN) & (s < P_MAX)]
    s = s[s.index.notna()].sort_index()
    return s[~s.index.duplicated()]

def download_station(stn):
    parts = []
    for yr in YEARS:
        path = f"data/ndbc/{stn}h{yr}.txt"
        if os.path.exists(path + ".missing"):
            continue
        if not os.path.exists(path):
            b = fetch(f"https://www.ndbc.noaa.gov/data/historical/stdmet/{stn}h{yr}.txt.gz")
            if b is None:
                open(path + ".missing", "w").close()
                continue
            with open(path, "w") as f:
                f.write(gzip.decompress(b).decode("latin-1"))
        with open(path) as f:
            s = parse_ndbc(f.read())
        if len(s):
            parts.append(s)
    if not parts:
        return pd.Series(dtype=float)
    s = pd.concat(parts).sort_index()
    return s[~s.index.duplicated()]

def to_hourly(raw):
    # Mean of the hour that just ended (label = end of hour): no peeking ahead.
    h = raw.resample("1h", label="right", closed="right").mean()
    full = pd.date_range(h.index.min(), h.index.max(), freq="1h")
    return h.reindex(full).interpolate(limit=2, limit_area="inside")

RAW = {}
for stn in STATION_POS:
    s = download_station(stn)
    if len(s):
        RAW[stn] = s
    print(f"{stn:6s} {len(s):>9,} readings  {s.index.year.nunique() if len(s) else 0:2d} years")
print(f"\n{len(RAW)} stations have pressure data")

## Step 4: storm tracks (HURDAT2)

HURDAT2 is NOAA's official record of every Atlantic storm: position, strength, and status every 6 hours. We interpolate it to every hour so we can measure the distance from each storm to each station at each hour.

Status codes: `TS` = tropical storm, `HU` = hurricane. Only those two count as "a storm is coming". If the download fails, download the file by hand from https://www.nhc.noaa.gov/data/#hurdat, upload it to Colab as `hurdat2.txt`, and re-run this cell.

In [ ]:
# STEP 4: HURDAT2 storm tracks
TROPICAL = {"TS", "HU"}

def get_hurdat_text():
    if os.path.exists("hurdat2.txt"):
        return open("hurdat2.txt", encoding="latin-1").read()
    urls = []
    listing = fetch("https://www.nhc.noaa.gov/data/hurdat/")
    if listing:
        found = set(re.findall(r"hurdat2-1851-(\d{4})-(\d{6})\.txt", listing.decode("latin-1")))
        if found:
            y, d = max(found, key=lambda t: (int(t[0]), t[1][4:], t[1][:4]))  # newest: year, then file date
            urls.append(f"https://www.nhc.noaa.gov/data/hurdat/hurdat2-1851-{y}-{d}.txt")
    urls.append(HURDAT_FALLBACK)
    for u in urls:
        b = fetch(u)
        if b:
            print("HURDAT2:", u)
            text = b.decode("latin-1")
            with open("hurdat2.txt", "w", encoding="latin-1") as f:
                f.write(text)
            return text
    raise RuntimeError("Download HURDAT2 by hand and upload it as hurdat2.txt")

def parse_hurdat(text, min_year):
    rows, sid, name = [], None, None
    for line in text.splitlines():
        p = [x.strip() for x in line.split(",")]
        if len(p) < 3:
            continue
        if not p[0].isdigit():            # header line: AL092022, IAN, 40,
            sid, name = p[0], p[1]
            continue
        if int(p[0][:4]) < min_year:
            continue
        lat = float(p[4][:-1]) * (1 if p[4][-1] == "N" else -1)
        lon = float(p[5][:-1]) * (1 if p[5][-1] == "E" else -1)
        rows.append((sid, name, p[0] + p[1].zfill(4), p[3], lat, lon))
    df = pd.DataFrame(rows, columns=["sid", "name", "ts", "status", "lat", "lon"])
    df["ts"] = pd.to_datetime(df.ts, format="%Y%m%d%H%M", utc=True)
    return df

def hourly_tracks(best):
    out = []
    for sid, g in best.groupby("sid"):
        g = g.drop_duplicates("ts").set_index("ts").sort_index()
        idx = pd.date_range(g.index.min().ceil("h"), g.index.max().floor("h"), freq="1h")
        if len(idx) == 0:
            continue
        gi = g.reindex(g.index.union(idx))
        gi[["lat", "lon"]] = gi[["lat", "lon"]].interpolate(method="time")
        gi["status"] = gi["status"].ffill()
        gi = gi.loc[idx]
        gi["sid"], gi["name"] = sid, g["name"].iloc[0]
        out.append(gi.rename_axis("ts").reset_index()[["sid", "name", "ts", "status", "lat", "lon"]])
    return pd.concat(out, ignore_index=True)

BEST = parse_hurdat(get_hurdat_text(), min_year=min(YEARS) - 1)
TRACKS = hourly_tracks(BEST)
print(f"{BEST.sid.nunique()} storms, {len(TRACKS):,} hourly track points")

## Step 5: labels + features

For every station and every hour:

**Label** = 1 if a TS/HU center comes within 150 km at any point in the **next 12 hours**, else 0.

**Features** (the box can compute every one of these from its own barometer):

| Feature | Meaning |
|---|---|
| `p_now` | pressure now (hPa, sea-level) |
| `d1`, `d3`, `d6` | change over the last 1, 3, 6 hours |
| `min6` | lowest pressure in the last 6 hours |
| `d3_rate` | how fast the 3-hour change is changing (is the drop speeding up?) |

We also mark **normal weather** hours (no storm within 500 km for ±48 h). False alarms are counted only there, so a risk alert during a real nearby storm isn't unfairly counted as "false".

We also record each **storm event**: the moment a storm was closest to a station. That's what recall and lead time are measured against.

In [ ]:
# STEP 5: labels, features, storm events
FEATURES = ["p_now", "d1", "d3", "d6", "min6", "d3_rate"]
FEATURE_HELP = {
    "p_now": "pressure now",
    "d1": "1-hour pressure change",
    "d3": "3-hour pressure change",
    "d6": "6-hour pressure change",
    "min6": "lowest pressure in the last 6 hours",
    "d3_rate": "speed-up of the 3-hour change",
}

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def build_station(stn, raw):
    lat, lon = STATION_POS[stn]
    df = pd.DataFrame({"p": to_hourly(raw)})
    df.index.name = "time"
    df["p_now"] = df.p
    df["d1"] = df.p.diff(1)
    df["d3"] = df.p.diff(3)
    df["d6"] = df.p.diff(6)
    df["min6"] = df.p.rolling(7).min()          # t-6h ... t
    df["d3_rate"] = df.d3.diff(1)

    tr = TRACKS.copy()
    tr["dist"] = haversine_km(lat, lon, tr.lat.values, tr.lon.values)
    near = tr[(tr.dist <= RADIUS_KM) & tr.status.isin(TROPICAL)]
    close = pd.Series(df.index.isin(near.ts).astype(float), index=df.index)
    # label at t = was the storm close at any hour in (t, t+12h]?
    df["label"] = close[::-1].rolling(HORIZON_H, min_periods=1).max()[::-1].shift(-1)
    wide = pd.Series(df.index.isin(tr[tr.dist <= NORMAL_KM].ts).astype(float), index=df.index)
    df["normal"] = wide.rolling(2 * 48 + 1, center=True, min_periods=1).max() == 0

    df["station"] = stn
    df["year"] = df.index.year
    df = df[df.index.month.isin(SEASON_MONTHS)].dropna(subset=FEATURES + ["label"])
    df["label"] = df.label.astype(int)

    events = []
    for sid, g in near.groupby("sid"):
        r = g.loc[g.dist.idxmin()]
        events.append(dict(station=stn, sid=sid, name=r["name"], year=r.ts.year,
                           t_closest=r.ts, min_dist_km=round(float(r.dist), 1)))
    return df, events

frames, all_events = [], []
for stn, raw in RAW.items():
    df, ev = build_station(stn, raw)
    frames.append(df)
    all_events += ev
DATA = pd.concat(frames).reset_index().sort_values(["station", "time"]).reset_index(drop=True)
EVENTS = pd.DataFrame(all_events)
print(f"{len(DATA):,} station-hours, {DATA.label.sum():,} labelled 'storm coming' ({100*DATA.label.mean():.2f}%)")
print(f"{len(EVENTS)} storm events (storm passed within {RADIUS_KM} km of a station)")
DATA.head()

## Step 6: split by year, then train both models

Storms are rare (well under 1% of hours), so both models use `class_weight="balanced"`: missing a storm hour costs as much as all the normal hours put together. Without that, a model could score 99% accuracy by always saying "no storm".

- **Logistic regression:** adds up weighted features → probability. Features are scaled first.
- **Decision tree (depth 3):** at most 3 yes/no questions about pressure. You can draw it.

In [ ]:
# STEP 6: split + train
def part(years):
    return DATA[DATA.year.isin(years)].reset_index(drop=True)

TRAIN, VAL, TEST = part(TRAIN_YEARS), part(VAL_YEARS), part(TEST_YEARS)
for n, d in [("train", TRAIN), ("val", VAL), ("test", TEST)]:
    print(f"{n:5s} {len(d):>9,} hours  {d.label.sum():>6,} storm-coming hours  "
          f"{EVENTS.year.isin(d.year.unique()).sum():>3} events")

X = lambda d: d[FEATURES].to_numpy(dtype=float)
MODELS = {
    "logreg": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)),
    "tree": DecisionTreeClassifier(max_depth=MAX_DEPTH, min_samples_leaf=MIN_LEAF,
                                   class_weight="balanced", random_state=0),
}
for name, m in MODELS.items():
    m.fit(X(TRAIN), TRAIN.label)
    print("trained", name)

## Step 7: pick each model's alarm threshold, then compare on unseen storms

This follows the rule in the project plan:

1. **Threshold** (on the validation years): the lowest threshold where false alarms in normal weather stay **≤ 1 per week**. A "false alarm" is the risk crossing the threshold (one episode, not every hour).
2. **Compare on the test years** (never seen):
   - **Event recall:** % of storms that triggered an alert in the 24 h before closest approach
   - **Lead time:** hours between the first alert and closest approach
3. **Winner:** better recall wins. **If the tree is within 5 points, ship the tree** because it can be explained.

In [ ]:
# STEP 7: thresholds + fair comparison
def false_alarms_per_week(df, alert):
    alert = np.asarray(alert, dtype=bool)
    same_run = ((df.station == df.station.shift(1)) &
                (df.time - df.time.shift(1) == pd.Timedelta("1h"))).to_numpy()
    prev = np.r_[False, alert[:-1]] & same_run
    onsets = alert & ~prev & df.normal.to_numpy()
    weeks = df.normal.sum() / 168
    return onsets.sum() / weeks if weeks else float("nan")

def event_metrics(df, alert, years):
    alert = np.asarray(alert, dtype=bool)
    by_stn = {s: (g.time.dt.tz_convert(None).to_numpy(), alert[g.index.to_numpy()])
              for s, g in df.groupby("station")}
    hour = np.timedelta64(1, "h")
    caught, leads, n = 0, [], 0
    for e in EVENTS[EVENTS.year.isin(years)].itertuples():
        if e.station not in by_stn:
            continue
        times, al = by_stn[e.station]
        tc = np.datetime64(e.t_closest.tz_convert(None))
        win = (times >= tc - 24 * hour) & (times <= tc)
        if win.sum() < 12:      # not enough data around this storm
            continue
        n += 1
        if al[win].any():
            caught += 1
            first = times[win][al[win]].min()
            leads.append((tc - first) / hour)
    return dict(events=n, recall=round(100 * caught / n, 1) if n else float("nan"),
                median_lead_h=round(float(np.median(leads)), 1) if leads else 0.0,
                mean_lead_h=round(float(np.mean(leads)), 1) if leads else 0.0)

def pick_threshold(model, df):
    prob = model.predict_proba(X(df))[:, 1]
    fa = float("nan")
    for thr in np.round(np.arange(0.05, 1.0, 0.01), 2):
        fa = false_alarms_per_week(df, prob >= thr)
        if fa <= FA_PER_WEEK_TARGET:
            return float(thr), fa
    return 0.99, fa

RESULTS = {}
for name, m in MODELS.items():
    thr, fa_val = pick_threshold(m, VAL)
    prob = m.predict_proba(X(TEST))[:, 1]
    alert = prob >= thr
    ev = event_metrics(TEST, alert, TEST_YEARS)
    lab = TEST.label.to_numpy() == 1
    RESULTS[name] = dict(
        threshold=thr,
        false_alarms_per_week_val=round(fa_val, 2),
        false_alarms_per_week_test=round(false_alarms_per_week(TEST, alert), 2),
        event_recall_pct=ev["recall"], events=ev["events"],
        median_lead_h=ev["median_lead_h"], mean_lead_h=ev["mean_lead_h"],
        hour_recall_pct=round(100 * (alert & lab).sum() / max(lab.sum(), 1), 1),
        hour_precision_pct=round(100 * (alert & lab).sum() / max(alert.sum(), 1), 1),
    )

COMPARE = pd.DataFrame(RESULTS).T
try:
    display(COMPARE)
except NameError:
    print(COMPARE)

lr_r, tr_r = RESULTS["logreg"]["event_recall_pct"], RESULTS["tree"]["event_recall_pct"]
WINNER = "tree" if (tr_r >= lr_r - 5) else "logreg"
print(f"\nWinner: {WINNER}  (tree recall {tr_r}% vs logistic regression {lr_r}%; tree wins ties within 5 points)")

## Step 8: pictures for the slides

- The decision tree itself (this is what the dashboard draws, with the current path highlighted)
- The comparison bar chart
- One real test storm: pressure and both risk curves, with the closest-approach time marked

In [ ]:
# STEP 8: figures
tree = MODELS["tree"]
print(export_text(tree, feature_names=FEATURES, decimals=2))

fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(tree, feature_names=FEATURES, class_names=["no storm", "storm coming"],
          filled=True, rounded=True, proportion=True, impurity=False, ax=ax, fontsize=9)
ax.set_title("StormBox storm risk tree (depth 3)")
fig.savefig("out/figures/decision_tree.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col, title in zip(axes, ["event_recall_pct", "median_lead_h", "false_alarms_per_week_test"],
                          ["Storms caught (%)", "Median lead time (h)", "False alarms / week"]):
    vals = [RESULTS[k][col] for k in ["logreg", "tree"]]
    ax.bar(["Logistic\nregression", "Decision\ntree"], vals, color=["#7a8ca3", "#2f6db5"])
    ax.set_title(title)
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v}", ha="center", va="bottom")
fig.suptitle("Test storms 2022+ (never seen in training)")
fig.tight_layout()
fig.savefig("out/figures/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

tev = EVENTS[EVENTS.year.isin(TEST_YEARS)].sort_values("min_dist_km")
for e in tev.itertuples():
    d = TEST[(TEST.station == e.station) &
             (TEST.time >= e.t_closest - pd.Timedelta("48h")) &
             (TEST.time <= e.t_closest + pd.Timedelta("24h"))]
    if len(d) < 48:
        continue
    fig, ax1 = plt.subplots(figsize=(11, 4))
    ax1.plot(d.time, d.p_now, color="black", label="pressure (hPa)")
    ax1.set_ylabel("hPa")
    ax2 = ax1.twinx()
    for name, color in [("logreg", "#7a8ca3"), ("tree", "#2f6db5")]:
        ax2.plot(d.time, MODELS[name].predict_proba(X(d))[:, 1], color=color, label=f"{name} risk")
        ax2.axhline(RESULTS[name]["threshold"], color=color, ls=":", lw=1)
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel("risk")
    ax1.axvline(e.t_closest, color="red", ls="--", label="closest approach")
    fig.legend(loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.08), frameon=False)
    ax1.set_title(f"{e.name.title()} {e.year} at station {e.station} ({e.min_dist_km} km away)")
    fig.savefig("out/figures/test_storm_example.png", dpi=150, bbox_inches="tight")
    plt.show()
    break

## Step 9: export a tiny model the Pi can run without scikit-learn

`storm_risk.json` holds **both** models (a few KB) plus which one to use (`"use"`). The dashboard reads the tree from it to draw the diagram.

`storm_risk_runtime.py` is ~80 lines of plain Python: it computes the features, predicts, and explains the answer in words for the agent. The cell checks that it gives exactly the same answers as scikit-learn.

**Important for the box:** NOAA pressure is **sea-level pressure**. The box's barometer reads **station pressure**, which is lower the higher up you are (about 1 hPa per 8 m). Convert first with `to_sea_level(reading, altitude_m)`. Changes (`d1`, `d3`…) barely care, but `p_now` and `min6` do.

**About the "%":** because of `class_weight="balanced"`, the risk number is a **score from 0 to 100**, not a calibrated chance. That's fine for an early-warning meter with a threshold; in the pitch say "risk score", not "78% chance".

In [ ]:
# STEP 9: export JSON model + pure-Python runtime
def tree_to_json(t):
    T = t.tree_
    nodes = []
    for i in range(T.node_count):
        v = T.value[i][0]
        node = {"id": i, "samples": int(T.n_node_samples[i]),
                "risk": float(v[1] / v.sum()) if v.sum() else 0.0}
        if T.children_left[i] != -1:
            node.update(feature=FEATURES[T.feature[i]], threshold=float(T.threshold[i]),
                        left=int(T.children_left[i]), right=int(T.children_right[i]))
        nodes.append(node)
    return {"nodes": nodes}

def logreg_to_json(pipe):
    sc, lr = pipe.named_steps["standardscaler"], pipe.named_steps["logisticregression"]
    return {"mean": sc.mean_.tolist(), "scale": sc.scale_.tolist(),
            "coef": lr.coef_[0].tolist(), "intercept": float(lr.intercept_[0])}

MODEL = {
    "version": "storm-risk-" + time.strftime("%Y%m%d"),
    "use": WINNER,
    "threshold": RESULTS[WINNER]["threshold"],
    "thresholds": {k: RESULTS[k]["threshold"] for k in RESULTS},
    "features": FEATURES,
    "feature_help": FEATURE_HELP,
    "label": f"TS/HU center within {RADIUS_KM} km in the next {HORIZON_H} h",
    "pressure_units": "hPa, sea-level (convert the box barometer with to_sea_level)",
    "tree": tree_to_json(MODELS["tree"]),
    "logreg": logreg_to_json(MODELS["logreg"]),
}
with open("out/models/storm_risk.json", "w") as f:
    json.dump(MODEL, f, indent=1)

METRICS = {
    "results_test": RESULTS, "winner": WINNER,
    "rule": "lowest threshold with <= 1 false alarm/week on validation years; tree wins if within 5 recall points",
    "train_years": [min(TRAIN_YEARS), max(TRAIN_YEARS)], "val_years": [min(VAL_YEARS), max(VAL_YEARS)],
    "test_years": [min(TEST_YEARS), max(TEST_YEARS)],
    "stations": sorted(RAW), "radius_km": RADIUS_KM, "horizon_h": HORIZON_H,
    "test_events": EVENTS[EVENTS.year.isin(TEST_YEARS)].assign(t_closest=lambda d: d.t_closest.astype(str)).to_dict("records"),
    "tree_text": export_text(MODELS["tree"], feature_names=FEATURES, decimals=2),
}
with open("out/models/metrics.json", "w") as f:
    json.dump(METRICS, f, indent=1, default=str)
joblib.dump(MODELS["tree"], "out/models/storm_risk_tree.joblib")
joblib.dump(MODELS["logreg"], "out/models/storm_risk_logreg.joblib")

RUNTIME = """\"\"\"StormBox storm-risk runtime: plain Python, no scikit-learn needed.

    model = load("models/storm_risk.json")
    feats = features_from_hourly(last_7_hourly_sea_level_pressures)   # oldest first
    result = predict(model, feats)      # {"risk": 0.78, "alert": True, ...}
    explain(model, result)              # "Storm risk 78% because ..."
\"\"\"
import json, math

def load(path="models/storm_risk.json"):
    with open(path) as f:
        return json.load(f)

def to_sea_level(p_hpa, altitude_m, temp_c=25.0):
    \"\"\"Barometer reading at the box -> sea-level pressure (what the model was trained on).\"\"\"
    if not altitude_m:
        return p_hpa
    return p_hpa * (1 - 0.0065 * altitude_m / (temp_c + 0.0065 * altitude_m + 273.15)) ** -5.257

def features_from_hourly(p):
    \"\"\"p: 7 hourly sea-level pressures, oldest first: [t-6h, t-5h, ..., now].\"\"\"
    p = list(p)[-7:]
    if len(p) < 7:
        raise ValueError("need 7 hourly values (t-6h ... now)")
    now = p[-1]
    d3 = now - p[-4]
    return {"p_now": now, "d1": now - p[-2], "d3": d3, "d6": now - p[0],
            "min6": min(p), "d3_rate": d3 - (p[-2] - p[-5])}

def predict(model, feats):
    return _tree(model, feats) if model["use"] == "tree" else _logreg(model, feats)

def _tree(model, feats):
    nodes, i, path = model["tree"]["nodes"], 0, []
    while "feature" in nodes[i]:
        n = nodes[i]
        x = feats[n["feature"]]
        left = x <= n["threshold"]
        path.append({"node": i, "feature": n["feature"], "value": x,
                     "threshold": n["threshold"], "op": "<=" if left else ">"})
        i = n["left"] if left else n["right"]
    risk = nodes[i]["risk"]
    return {"model": "tree", "risk": risk, "alert": risk >= model["threshold"],
            "path": path, "leaf": i}

def _logreg(model, feats):
    m = model["logreg"]
    z, parts = m["intercept"], []
    for f, mu, sd, w in zip(model["features"], m["mean"], m["scale"], m["coef"]):
        c = w * (feats[f] - mu) / sd
        z += c
        parts.append((f, c))
    z = max(-30.0, min(30.0, z))
    risk = 1 / (1 + math.exp(-z))
    return {"model": "logreg", "risk": risk, "alert": risk >= model["threshold"],
            "top_factors": sorted(parts, key=lambda t: -abs(t[1]))[:3]}

def explain(model, result):
    \"\"\"One plain-English sentence the agent can say or the dashboard can show.\"\"\"
    h = model["feature_help"]
    pct = round(100 * result["risk"])
    if "path" in result:
        last = {}                       # keep the tightest check per feature
        for s in result["path"]:
            last[s["feature"]] = s
        why = [f"{h[f]} is {s['value']:.1f} hPa "
               f"({'at or below' if s['op'] == '<=' else 'above'} {s['threshold']:.1f})"
               for f, s in last.items()]
    else:
        why = [f"{h[f]} {'raises' if c > 0 else 'lowers'} the risk" for f, c in result["top_factors"]]
    return f"Storm risk {pct}% because " + "; ".join(why) + "."
"""
with open("out/models/storm_risk_runtime.py", "w") as f:
    f.write(RUNTIME)

# Check: the runtime must match scikit-learn exactly
rt = {}
exec(RUNTIME, rt)
sample = TEST.sample(min(3000, len(TEST)), random_state=0)
for name in ["tree", "logreg"]:
    mdl = dict(MODEL, use=name)
    ours = np.array([rt["predict"](mdl, r)["risk"] for r in sample[FEATURES].to_dict("records")])
    theirs = MODELS[name].predict_proba(X(sample))[:, 1]
    assert np.allclose(ours, theirs, atol=1e-6), name
print("Runtime matches scikit-learn for both models")

top = sample.assign(r=MODELS[WINNER].predict_proba(X(sample))[:, 1]).sort_values("r").iloc[-1]
print("Example:", rt["explain"](MODEL, rt["predict"](MODEL, top[FEATURES].to_dict())))
print("Model file size:", os.path.getsize("out/models/storm_risk.json"), "bytes")

## Step 10: real storms for the demo replay

For each test storm, this takes the station that got closest and saves 36 h before to 18 h after closest approach, one reading every 10 minutes: `ts,pressure_hpa`. That matches §6.8 of the plan (1 reading = 10 min, ~54 h ≈ 324 rows; at 4 readings/s that's ~80 s, or speed it up).

`replay/index.json` lists each file with the storm name, station, and closest distance for the dashboard menu.

In [ ]:
# STEP 10: replay CSVs
def export_replays(max_storms=6, before_h=36, after_h=18):
    index, done = [], set()
    tev = EVENTS[EVENTS.year.isin(TEST_YEARS)].sort_values("min_dist_km")
    for e in tev.itertuples():
        if e.sid in done or e.station not in RAW or len(done) >= max_storms:
            continue
        start, end = e.t_closest - pd.Timedelta(hours=before_h), e.t_closest + pd.Timedelta(hours=after_h)
        seg = RAW[e.station][start - pd.Timedelta("1h"): end + pd.Timedelta("1h")]
        grid = pd.date_range(start.ceil("10min"), end.floor("10min"), freq="10min")
        s = seg.reindex(seg.index.union(grid)).interpolate(method="time", limit=6, limit_area="inside").reindex(grid)
        if s.notna().mean() < 0.9:
            continue
        s = s.interpolate(limit_direction="both").round(1)
        fname = f"{e.name.lower()}_{e.year}_{e.station}.csv"
        pd.DataFrame({"ts": s.index.strftime("%Y-%m-%dT%H:%M:%SZ"), "pressure_hpa": s.values}) \
          .to_csv(f"out/replay/{fname}", index=False)
        done.add(e.sid)
        index.append(dict(file=fname, storm=e.name.title(), year=int(e.year), station=e.station,
                          closest_km=e.min_dist_km, closest_time=str(e.t_closest),
                          min_pressure_hpa=float(s.min()), rows=len(s)))
        print(f"{fname:32s} min {s.min():.1f} hPa, {e.min_dist_km} km away")
    with open("out/replay/index.json", "w") as f:
        json.dump(index, f, indent=1)
    return index

REPLAYS = export_replays()

## Step 11: download everything

The zip contains:

| File | Goes to (repo, §13) |
|---|---|
| `models/storm_risk.json` | `models/` (the Pi loads this) |
| `models/storm_risk_runtime.py` | `pi/box_service/` |
| `models/metrics.json` | `models/` (engineer-mode model panel) |
| `models/*.joblib` | `ml/` (backup only; the Pi doesn't need them) |
| `replay/*.csv`, `replay/index.json` | `replay/` |
| `figures/*.png` | `docs/` (slides) |

In [ ]:
# STEP 11: zip + download
shutil.make_archive("stormbox_storm_model", "zip", "out")
print("Created stormbox_storm_model.zip")
try:
    from google.colab import files
    files.download("stormbox_storm_model.zip")
except ImportError:
    print("Not running in Colab: the zip is next to this notebook.")

## What to try if the results look bad

| Problem | Try |
|---|---|
| Too many false alarms on test | Lower `FA_PER_WEEK_TARGET` (e.g. 0.5) |
| Tree catches few storms | `MAX_DEPTH = 4`, or `RADIUS_KM = 200`, or `HORIZON_H = 18` |
| Very few test events | Add more stations to `STATIONS` |
| A station has no data | Fine, it's skipped. Check the station ID on ndbc.noaa.gov |

Whatever you change, write it down: the comparison table + tree picture **is** your ML slide.